# Setting Up All Artifacts details

In [ ]:
!pip install super-gradients==3.1.2 

In [ ]:
## Give appropriate permission to the directory "FOLDER_WITH_ARTIFACTS" you are working with
import os
os.environ['QNN_TARGET_ARCH'] = "aarch64-android"

In [ ]:
## Note- Use python3.8 or above for generating onnx
import torch
from super_gradients.training import models
from super_gradients.common.object_names import Models
import cv2
import numpy as np
import os

## Getting the ONNX Model

In [ ]:
os.makedirs('models', exist_ok=True)

In [ ]:
import pathlib, sys, re

SG_ROOT = pathlib.Path(sys.executable).parent.parent / "lib" / "python3.10" / "site-packages" / "super_gradients"

NEW_HOST = "https://d2gjn4b69gu75n.cloudfront.net/models/"

def _patch_file(path: pathlib.Path, old: str, new: str, label: str) -> None:
    if not path.exists():
        print(f"SKIP  {label}: file not found ({path})")
        return
    text = path.read_text()
    if new in text:
        print(f"OK    {label}: already patched")
    elif old in text:
        path.write_text(text.replace(old, new))
        print(f"FIXED {label}")
    else:
        print(f"WARN  {label}: expected pattern not found — manual check needed")

# Fix 1: update download URL in pretrained_models.py
# Replaces any previously set host (sghub.deci.ai or sg-hub-nv.s3.amazonaws.com)
pretrained_path = SG_ROOT / "training" / "pretrained_models.py"
if pretrained_path.exists():
    text = pretrained_path.read_text()
    patched, n = re.subn(
        r'"yolo_nas_s_coco":\s*"https://[^"]+/yolo_nas_s_coco\.pth"',
        f'"yolo_nas_s_coco": "{NEW_HOST}yolo_nas_s_coco.pth"',
        text,
    )
    if f'"{NEW_HOST}yolo_nas_s_coco.pth"' in text:
        print("OK    pretrained_models.py – yolo_nas_s_coco URL: already patched")
    elif n:
        pretrained_path.write_text(patched)
        print("FIXED pretrained_models.py – yolo_nas_s_coco URL")
    else:
        print("WARN  pretrained_models.py – yolo_nas_s_coco URL: pattern not found")
else:
    print(f"SKIP  pretrained_models.py: file not found ({pretrained_path})")

# Fix 2: update URL used to derive unique cache filename in checkpoint_utils.py
checkpoint_path = SG_ROOT / "training" / "utils" / "checkpoint_utils.py"
if checkpoint_path.exists():
    text = checkpoint_path.read_text()
    patched, n = re.subn(
        r'url\.split\("https://[^"]+/models/"\)\[1\]',
        f'url.split("{NEW_HOST}")[1]',
        text,
    )
    if f'url.split("{NEW_HOST}")[1]' in text:
        print("OK    checkpoint_utils.py – unique_filename URL split: already patched")
    elif n:
        checkpoint_path.write_text(patched)
        print("FIXED checkpoint_utils.py – unique_filename URL split")
    else:
        print("WARN  checkpoint_utils.py – unique_filename URL split: pattern not found")
else:
    print(f"SKIP  checkpoint_utils.py: file not found ({checkpoint_path})")

# Fix 3: fix circular import in training/__init__.py
_patch_file(
    SG_ROOT / "training" / "__init__.py",
    "import super_gradients.training.utils.distributed_training_utils as distributed_training_utils",
    "from .utils import distributed_training_utils",
    "training/__init__.py – circular import",
)

# Clear any cached super_gradients modules so patched files take effect on next import
evicted = [k for k in list(sys.modules) if "super_gradients" in k]
for k in evicted:
    del sys.modules[k]
if evicted:
    print(f"Cleared {len(evicted)} cached super_gradients module(s) — patches will take effect on next import")
else:
    print("super_gradients not yet imported — patches will take effect on first import")

In [ ]:
model = models.get(Models.YOLO_NAS_S, pretrained_weights="coco")
# Prpare model for conversion
# Input size is in format of [Batch x Channels x Width x Height] where 640 is the standard COCO dataset dimensions
model.eval()
model.prep_model_for_conversion(input_size=[1, 3, 320, 320])
# Create dummy_input
dummy_input = torch.randn([1, 3, 320, 320], device="cpu")
# Convert model to onnx
torch.onnx.export(model, dummy_input, "models/yolo_nas_s.onnx", opset_version=11)

#### Getting the FP32 Model

In [ ]:
%%bash
source $QAIRT_SDK_ROOT/bin/envsetup.sh
export PATH=${ANDROID_NDK_ROOT}:${PATH}
qnn-onnx-converter --input_network models/yolo_nas_s.onnx --output_path output_cpu/yolo_nas_s.cpp --out_node 885 --out_node 877
qnn-model-lib-generator -c output_cpu/yolo_nas_s.cpp -b output_cpu/yolo_nas_s.bin --output_dir output_cpu/ -t aarch64-android
mv output_cpu/aarch64-android/libyolo_nas_s.so  output_cpu/aarch64-android/libyolo_nas_w8a8.so

### Getting The dataset
Please, fill coco dataset link in below code block. You might need 10-15 images for quantization.

In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip -q --show-progress
!unzip val2017.zip
!mkdir "raw"

In [ ]:
files = os.listdir('val2017') #val2017 is the datatset folder path. Keeping only 15 images.
for file in files[15:]:
    os.remove("val2017/"+file)

In [ ]:
%%bash
rm -rf val2017.zip

#### Getting the Quatized Model for DSP

In [ ]:
import cv2
import numpy as np
import os

def preprocess(original_image):
    resized_image = cv2.resize(original_image, (320, 320))
    resized_image = resized_image/255
    return resized_image

##Please download Coco2014 dataset and give the path here
dataset_path = "val2017/"

filenames=[]
for path in os.listdir(dataset_path):
    # check if current path is a file
    if os.path.isfile(os.path.join(dataset_path, path)):
        filenames.append(os.path.join(dataset_path, path))
        
for filename in filenames:
    original_image = cv2.imread(filename)
    img = preprocess(original_image)
    img = img.astype(np.float32)
    img.tofile("raw/"+filename.split("/")[-1].split(".")[0]+".raw")


In [ ]:
%%bash
find raw -name *.raw > input.txt
cat input.txt

In [ ]:
%%bash
source $QAIRT_SDK_ROOT/bin/envsetup.sh
export PATH=${ANDROID_NDK_ROOT}:${PATH}
qnn-onnx-converter --input_network models/yolo_nas_s.onnx --output_path output_dsp/yolo_nas_s_quantized.cpp --out_node 885 --out_node 877 --input_list input.txt
qnn-model-lib-generator -c output_dsp/yolo_nas_s_quantized.cpp -b output_dsp/yolo_nas_s_quantized.bin --output_dir output_dsp/ -t aarch64-android
mv output_dsp/aarch64-android/libyolo_nas_s_quantized.so  output_dsp/aarch64-android/libyolo_nas_w8a8_dsp.so

## Setting up Android App Dependencies

Copies OpenCV Android SDK and QAIRT/SNPE native libraries into the app source tree so the Android project can be built in Android Studio.

In [ ]:
%%bash

bash resolveDependencies.sh